# 01 · Data Preparation & Target Definition

**Objective:** build a clean, analysis-ready loan population and a single, documented definition of a "bad" loan that every later analysis reuses.

**Bottom line:** the raw file holds 2,260,701 accepted loans (2007–2018). After removing 33 blank rows, I restrict the performance population to **445,743 36-month loans issued in 2014–2015**: every one of them had reached maturity by the end of the data (Dec-2018), so **99.98% have a final outcome**. Using this population, the portfolio **bad rate is 14.5%**.

| Step | Rows | Note |
|---|---|---|
| Raw accepted loans | 2,260,701 | 151 columns |
| Remove blank rows | 2,260,668 | 33 rows with every column empty |
| 2014–2015, 36-month cohort | 445,743 | matured by Dec-2018 |
| Indeterminate (excluded from rates) | 86 | Current / In Grace / Late 16–30 |

## 1. Setup
Raw CSVs are converted to Parquet once (columnar storage: ~10× smaller, queries read only the columns they need). All views are defined in [`SQL/00_create_views.sql`](../SQL/00_create_views.sql).

In [1]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from lending_db import get_connection, IMAGE_DIR

con = get_connection()
pd.set_option("display.max_columns", None)
plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

## 2. Data profiling

In [2]:
con.sql("""
SELECT COUNT(*)                         AS n_rows,
       MIN(strptime(issue_d, '%b-%Y'))  AS first_issue,
       MAX(strptime(issue_d, '%b-%Y'))  AS last_issue
FROM loans
""").df()

,n_rows,first_issue,last_issue
0,2260701,2007-06-01,2018-12-01


In [3]:
# Loan status distribution across the full book
con.sql("""
SELECT loan_status,
       COUNT(*)                                        AS n_loans,
       ROUND(COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (), 4) AS pct_loans
FROM loans
GROUP BY loan_status
ORDER BY n_loans DESC
""").df()

,loan_status,n_loans,pct_loans
0,Fully Paid,1076751,0.4763
1,Current,878317,0.3885
2,Charged Off,268559,0.1188
3,Late (31-120 days),21467,0.0095
4,In Grace Period,8436,0.0037
5,Late (16-30 days),4349,0.0019
6,Does not meet the credit policy. Status:Fully ...,1988,0.0009
7,Does not meet the credit policy. Status:Charge...,761,0.0003
8,Default,40,0.0000
9,NaN,33,0.0000


## 3. Data quality issues found and fixed

| # | Issue | How it was found | Fix |
|---|---|---|---|
| 1 | 33 rows with `loan_status` NULL and **every** column empty (file footer / blank rows) | `WHERE loan_status IS NULL` | Dropped (they are not loans) |
| 2 | `issue_d` stored as text (`'Dec-2015'`); text sorting puts `Apr-2015` before `Dec-2014` | `SELECT DISTINCT issue_d` | `strptime(issue_d, '%b-%Y')` → `issue_date` |
| 3 | `term = '36 months'` returned **0 rows** | Result failed a sanity check; `LENGTH(term)` = 10, not 9 | Values carry a leading space (`' 36 months'`) → `TRIM` and cast to integer `term_months` |

In [4]:
# Issue 1: blank rows
con.sql("""
SELECT COUNT(*) AS n_rows,
       COUNT(id) AS n_with_id, COUNT(loan_amnt) AS n_with_amount, COUNT(grade) AS n_with_grade
FROM loans
WHERE loan_status IS NULL
""").df()

,n_rows,n_with_id,n_with_amount,n_with_grade
0,33,33,0,0


In [5]:
# Issue 3: invisible leading space in `term`
con.sql("""
SELECT term, LENGTH(term) AS term_length, COUNT(*) AS n_loans
FROM loans_clean
GROUP BY term
ORDER BY term
""").df()

,term,term_length,n_loans
0,36 months,10,1609754
1,60 months,10,650914


## 4. Performance window: why 2014–2015, 36-month loans

The data ends in **Dec-2018**. A 36-month loan issued in Dec-2015 matures in Dec-2018, so every 36-month loan issued in 2014–2015 has a **final outcome**. Newer loans and 60-month loans are mostly still `Current`:

* treating them as good would **understate the bad rate**;
* dropping only the unfinished ones would bias the sample toward loans that **defaulted early** (right-censoring).

In [6]:
con.sql("""
SELECT YEAR(issue_date)                                   AS issue_year,
       term_months,
       COUNT(*)                                           AS n_loans,
       ROUND(AVG(CASE WHEN loan_status = 'Current' THEN 1.0 ELSE 0 END), 4) AS pct_still_current
FROM loans_clean
WHERE YEAR(issue_date) BETWEEN 2013 AND 2018
GROUP BY issue_year, term_months
ORDER BY issue_year, term_months
""").df()

,issue_year,term_months,n_loans,pct_still_current
0,2013,36,100422,0.0000
1,2013,60,34392,0.0002
2,2014,36,162570,0.0000
3,2014,60,73059,0.1631
4,2015,36,283173,0.0003
5,2015,60,137922,0.3134
6,2016,36,323495,0.2678
7,2016,60,110912,0.4277
8,2017,36,320419,0.5741
9,2017,60,123160,0.6325


## 5. Target definition (`bad_flag`)

Industry-standard **good / bad / indeterminate** split. Indeterminate loans get `NULL`, so `AVG(bad_flag)` excludes them automatically.

| loan_status | bad_flag | Rationale |
|---|---|---|
| Charged Off | 1 | Written off |
| Late (31–120 days) | 1 | Seriously delinquent; very low cure rate |
| Fully Paid | 0 | Repaid in full |
| Current, In Grace Period, Late (16–30 days) | NULL | Outcome not yet known |

In [7]:
con.sql("""
SELECT loan_status,
       bad_flag,
       COUNT(*) AS n_loans
FROM cohort
GROUP BY loan_status, bad_flag
ORDER BY n_loans DESC
""").df()

,loan_status,bad_flag,n_loans
0,Fully Paid,0,381149
1,Charged Off,1,64447
2,Current,<NA>,72
3,Late (31-120 days),1,61
4,In Grace Period,<NA>,9
5,Late (16-30 days),<NA>,5


In [8]:
con.sql("""
SELECT COUNT(*)                     AS n_loans,
       COUNT(bad_flag)              AS n_with_outcome,
       COUNT(*) - COUNT(bad_flag)   AS n_indeterminate,
       ROUND(AVG(bad_flag), 4)      AS bad_rate
FROM cohort
""").df()

,n_loans,n_with_outcome,n_indeterminate,bad_rate
0,445743,445657,86,0.1447
